# Phase 5 — Resultats Finaux
Comparaison complete : Baseline vs Attaques vs Defenses.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import random
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('libs OK')

libs OK


## 1. Chargement des donnees et modeles

In [4]:
def preprocess(train_df, test_df):
    drop_cols = ['id', 'attack_cat']
    train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    test_df  = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    for df in [train_df, test_df]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
    median_vals = train_df.median(numeric_only=True)
    train_df.fillna(median_vals, inplace=True)
    test_df.fillna(median_vals, inplace=True)
    cat_cols = [c for c in train_df.select_dtypes(include=['object','str']).columns if c != 'label']
    le = LabelEncoder()
    for col in cat_cols:
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        known = set(le.classes_)
        test_df[col] = test_df[col].astype(str).apply(lambda x: x if x in known else le.classes_[0])
        test_df[col] = le.transform(test_df[col])
    X_train = train_df.drop('label', axis=1).values.astype(np.float32)
    y_train = train_df['label'].values.astype(np.float32)
    X_test  = test_df.drop('label', axis=1).values.astype(np.float32)
    y_test  = test_df['label'].values.astype(np.float32)
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    return X_train, y_train, X_test, y_test, scaler

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1),          nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

train_df = pd.read_csv('../data/UNSW_NB15_training-set.csv')
test_df  = pd.read_csv('../data/UNSW_NB15_testing-set.csv')
X_train, y_train, X_test, y_test, scaler = preprocess(train_df.copy(), test_df.copy())

input_dim = X_train.shape[1]
model_baseline = MLP(input_dim)
model_baseline.load_state_dict(torch.load('../results/baseline_model.pth', weights_only=True))
model_baseline.eval()

model_adv = MLP(input_dim)
model_adv.load_state_dict(torch.load('../results/adversarial_model.pth', weights_only=True))
model_adv.eval()

cols = pd.read_csv('../data/UNSW_NB15_training-set.csv').drop(columns=['id','attack_cat','label']).columns.tolist()
MANIPULABLE = ['dur','spkts','dpkts','sbytes','dbytes','sttl','sload','dload','sinpkt','dinpkt','sjit','djit','smean','dmean']
manipulable_idx = [cols.index(f) for f in MANIPULABLE if f in cols]
mask = torch.zeros(len(cols))
mask[manipulable_idx] = 1.0

print('Modeles charges OK')

FileNotFoundError: [Errno 2] No such file or directory: '../results/adversarial_model.pth'

## 2. Fonctions utilitaires

In [ ]:
def fgsm_attack(model, X, y, epsilon, mask):
    X_t = torch.FloatTensor(X).requires_grad_(True)
    loss = nn.BCELoss()(model(X_t).squeeze(), torch.FloatTensor(y))
    loss.backward()
    return (X_t + epsilon * X_t.grad.sign() * mask).detach().numpy()

def pgd_attack(model, X, y, epsilon, alpha, steps, mask):
    X_orig = torch.FloatTensor(X)
    X_adv  = X_orig.clone()
    y_t    = torch.FloatTensor(y)
    for _ in range(steps):
        X_adv = X_adv.detach().requires_grad_(True)
        loss  = nn.BCELoss()(model(X_adv).squeeze(), y_t)
        loss.backward()
        X_adv = X_orig + torch.clamp(X_adv + alpha * X_adv.grad.sign() * mask - X_orig, -epsilon, epsilon)
    return X_adv.detach().numpy()

def feature_squeezing(X, n_bits=4):
    max_val = np.max(np.abs(X), axis=0, keepdims=True) + 1e-8
    X_norm  = X / max_val
    return np.round(X_norm * (2**n_bits)) / (2**n_bits) * max_val

def evaluate(model, X, y_true):
    model.eval()
    with torch.no_grad():
        prob = model(torch.FloatTensor(X)).squeeze().numpy()
        pred = (prob >= 0.5).astype(int)
    evasion = (pred[y_true == 1] == 0).mean() * 100
    return {
        'F1':        f1_score(y_true, pred),
        'Precision': precision_score(y_true, pred),
        'Recall':    recall_score(y_true, pred),
        'PR-AUC':    average_precision_score(y_true, prob),
        'Evasion%':  evasion
    }

attack_idx = np.where(y_test == 1)[0]
X_attacks  = X_test[attack_idx]
y_attacks  = y_test[attack_idx]
EPS        = 0.1

# Generer toutes les variantes adversariales
def make_adv(model, eps):
    X_f = X_test.copy(); X_f[attack_idx] = fgsm_attack(model, X_attacks, y_attacks, eps, mask)
    X_p = X_test.copy(); X_p[attack_idx] = pgd_attack(model, X_attacks, y_attacks, eps, 0.01, 40, mask)
    return X_f, X_p

X_fgsm_b, X_pgd_b = make_adv(model_baseline, EPS)
X_fgsm_a, X_pgd_a = make_adv(model_adv, EPS)
X_sq      = feature_squeezing(X_test)
X_fgsm_sq = feature_squeezing(X_fgsm_b)
X_pgd_sq  = feature_squeezing(X_pgd_b)
print('Toutes les variantes generees OK')

## 3. Tableau de resultats complet

In [ ]:
rows = [
    ('Baseline',          'Propres', evaluate(model_baseline, X_test,    y_test)),
    ('Baseline',          'FGSM',    evaluate(model_baseline, X_fgsm_b,  y_test)),
    ('Baseline',          'PGD',     evaluate(model_baseline, X_pgd_b,   y_test)),
    ('Adv. Training',     'Propres', evaluate(model_adv,      X_test,    y_test)),
    ('Adv. Training',     'FGSM',    evaluate(model_adv,      X_fgsm_a,  y_test)),
    ('Adv. Training',     'PGD',     evaluate(model_adv,      X_pgd_a,   y_test)),
    ('Feat. Squeezing',   'Propres', evaluate(model_baseline, X_sq,      y_test)),
    ('Feat. Squeezing',   'FGSM',    evaluate(model_baseline, X_fgsm_sq, y_test)),
    ('Feat. Squeezing',   'PGD',     evaluate(model_baseline, X_pgd_sq,  y_test)),
]

df_results = pd.DataFrame([
    {'Modele': m, 'Scenario': s, 'F1': r['F1'], 'Precision': r['Precision'],
     'Recall': r['Recall'], 'PR-AUC': r['PR-AUC'], 'Evasion%': r['Evasion%']}
    for m, s, r in rows
])

print(df_results.to_string(index=False, float_format='{:.4f}'.format))
df_results.to_csv('../results/final_results.csv', index=False)
print('\nSauvegarde : results/final_results.csv')

## 4. Graphique comparatif — F1-score

In [ ]:
scenarios = ['Propres', 'FGSM', 'PGD']
models_names = ['Baseline', 'Adv. Training', 'Feat. Squeezing']
colors = ['steelblue', 'tomato', 'seagreen']
x = np.arange(len(scenarios))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# F1-score
for i, (name, color) in enumerate(zip(models_names, colors)):
    vals = [r['F1'] for m, s, r in rows if m == name]
    bars = ax1.bar(x + (i-1)*width, vals, width, label=name, color=color)
    ax1.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)
ax1.set_ylabel('F1-score')
ax1.set_title('F1-score : Baseline vs Defenses')
ax1.set_xticks(x)
ax1.set_xticklabels(scenarios)
ax1.legend()
ax1.set_ylim(0, 1.1)

# Evasion Rate
for i, (name, color) in enumerate(zip(models_names, colors)):
    vals = [r['Evasion%'] for m, s, r in rows if m == name]
    bars = ax2.bar(x + (i-1)*width, vals, width, label=name, color=color)
    ax2.bar_label(bars, fmt='%.1f%%', padding=2, fontsize=8)
ax2.set_ylabel('Evasion Rate (%)')
ax2.set_title('Evasion Rate : moins = mieux')
ax2.set_xticks(x)
ax2.set_xticklabels(scenarios)
ax2.legend()

plt.tight_layout()
plt.savefig('../results/final_comparison.png', dpi=150)
plt.show()
print('Sauvegarde : results/final_comparison.png')

## 5. Matrices de confusion

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
configs = [
    (model_baseline, X_test,    'Baseline / Propres'),
    (model_baseline, X_fgsm_b,  'Baseline / FGSM'),
    (model_baseline, X_pgd_b,   'Baseline / PGD'),
    (model_adv,      X_test,    'Adv.Training / Propres'),
    (model_adv,      X_fgsm_a,  'Adv.Training / FGSM'),
    (model_adv,      X_pgd_a,   'Adv.Training / PGD'),
]
for ax, (mdl, X, title) in zip(axes.flatten(), configs):
    with torch.no_grad():
        pred = (mdl(torch.FloatTensor(X)).squeeze().numpy() >= 0.5).astype(int)
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Normal','Attaque']).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150)
plt.show()
print('Sauvegarde : results/confusion_matrices.png')

## 6. Robustesse vs epsilon (variation experimentale)

In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.2, 0.3]
f1_b, f1_a, ev_b, ev_a = [], [], [], []

for eps in epsilons:
    Xf_b = X_test.copy(); Xf_b[attack_idx] = pgd_attack(model_baseline, X_attacks, y_attacks, eps, 0.01, 20, mask)
    Xf_a = X_test.copy(); Xf_a[attack_idx] = pgd_attack(model_adv,      X_attacks, y_attacks, eps, 0.01, 20, mask)
    r_b = evaluate(model_baseline, Xf_b, y_test)
    r_a = evaluate(model_adv,      Xf_a, y_test)
    f1_b.append(r_b['F1']);      ev_b.append(r_b['Evasion%'])
    f1_a.append(r_a['F1']);      ev_a.append(r_a['Evasion%'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(epsilons, f1_b, 'o-', color='steelblue', label='Baseline')
ax1.plot(epsilons, f1_a, 's-', color='tomato',    label='Adv. Training')
ax1.set_title('F1-score vs budget epsilon (PGD)')
ax1.set_xlabel('Epsilon'); ax1.set_ylabel('F1-score'); ax1.legend()

ax2.plot(epsilons, ev_b, 'o-', color='steelblue', label='Baseline')
ax2.plot(epsilons, ev_a, 's-', color='tomato',    label='Adv. Training')
ax2.set_title('Evasion Rate vs budget epsilon (PGD)')
ax2.set_xlabel('Epsilon'); ax2.set_ylabel('Evasion Rate (%)'); ax2.legend()

plt.tight_layout()
plt.savefig('../results/robustness_vs_epsilon.png', dpi=150)
plt.show()
print('Sauvegarde : results/robustness_vs_epsilon.png')
print('\n=== PROJET TERMINE ===')
print('Resultats dans results/final_results.csv')
print('Graphiques dans results/')